In [2]:
:dep plotters = { version = "0.3", default-features = false, features = ["evcxr", "all_series"] }

In [3]:
fn step(u: &[f64], r: f64) -> Vec<f64> {
    let n = u.len();
    (0..n)
        .map(|i| match i {
            0 => 0.0,
            i if i == n - 1 => 0.0,
            i => u[i] + r * (u[i - 1] - 2.0 * u[i] + u[i + 1]),
        })
        .collect()
}

fn solve_at(u0: &[f64], r: f64, steps: usize) -> Vec<f64> {
    (0..steps).fold(u0.to_vec(), |u, _| step(&u, r))
}

let n = 101;
let alpha = 1.0;
let dx = 1.0 / (n - 1) as f64;
let dt = 0.4 * dx * dx / alpha;          // explicit scheme is stable when r <= 0.5
let r = alpha * dt / (dx * dx);
let xs: Vec<f64> = (0..n).map(|i| i as f64 * dx).collect();
let u0: Vec<f64> = xs.iter().map(|x| (std::f64::consts::PI * x).sin()).collect();

In [8]:
use plotters::prelude::*;

let times = [0.0, 0.02, 0.05, 0.1, 0.2];

evcxr_figure((720, 440), |root| {
    root.fill(&WHITE)?;
    let mut chart = ChartBuilder::on(&root)
        .caption("Heat equation: u(x, t)", ("sans-serif", 22))
        .margin(10)
        .x_label_area_size(35)
        .y_label_area_size(45)
        .build_cartesian_2d(0f64..1f64, 0f64..1.05f64)?;
    chart.configure_mesh().x_desc("x").y_desc("u").draw()?;

    for (k, &t) in times.iter().enumerate() {
        let color = Palette99::pick(k);
        let u = solve_at(&u0, r, (t / dt).round() as usize);

        chart
            .draw_series(LineSeries::new(xs.iter().copied().zip(u), color.stroke_width(2)))?
            .label(format!("t = {t}"))
            .legend(move |(x, y)| PathElement::new([(x, y), (x + 20, y)], &color));

        // exact solution as dots, to check the numerics
        let exact = xs.iter().step_by(10).map(|&x| {
            (x, (-alpha * std::f64::consts::PI.powi(2) * t).exp() * (std::f64::consts::PI * x).sin())
        });
        chart.draw_series(exact.map(|p| Circle::new(p, 3, color.filled())))?;
    }

    chart.configure_series_labels().border_style(&BLACK).background_style(&WHITE).draw()?;
    Ok(())
})

Error: borrow of moved value: `color`